# HY562 / CS-562 — Environment check (Fall 2026)
Run all cells (**Run ▸ Run All Cells**). Every cell should print ✅. If something prints ❌, see
`SETUP_GUIDE.md` → *Troubleshooting*.

In [ ]:
import sys, platform, subprocess, shutil, os
print("Python   :", sys.version.split()[0], "|", platform.platform())
java = shutil.which("java")
if java:
    out = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    print("Java     :", [l for l in out.splitlines() if "version" in l][0])
    print("✅ Java found at", java)
else:
    print("❌ Java not found on PATH (Spark needs Java 17 or 21)")

In [ ]:
import pyspark
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder
         .appName("hy562-env-check")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
print("PySpark  :", pyspark.__version__)
print("Spark    :", spark.version)
print("Scala    :", sc._jvm.scala.util.Properties.versionString())
print("JVM      :", sc._jvm.java.lang.System.getProperty("java.version"))
print("Spark UI :", sc.uiWebUrl, "(open http://localhost:4040 from your browser when using Docker)")
print("✅ SparkSession started" if spark.version.startswith("4.") else "❌ expected Spark 4.x")

In [ ]:
# a tiny DataFrame job: word count on three lines
df = spark.createDataFrame([("to be or not to be",), ("that is the question",), ("to be is to do",)], ["line"])
wc = (df.select(F.explode(F.split("line", " ")).alias("word"))
        .groupBy("word").count().orderBy(F.desc("count"), "word"))
wc.show(5)
assert wc.first()["word"] == "to"
print("✅ DataFrame job OK")

In [ ]:
# course data folder (the notebooks use relative paths ../05_Data or ../../05_Data)
from pathlib import Path
data = next((p for p in [Path("../05_Data"), Path("../../05_Data"), Path("05_Data")] if p.exists()), None)
if data:
    print("✅ data folder:", data.resolve(), sorted(x.name for x in data.iterdir()))
    if (data / "retail.parquet").exists():
        r = spark.read.parquet(str(data / "retail.parquet"))
        print("retail.parquet rows:", r.count(), "| transactions:", r.select("tid").distinct().count())
else:
    print("⚠️ 05_Data folder not found next to this notebook (only needed for the labs)")

In [ ]:
# Kafka reachability (only inside the Docker environment; skipped gracefully otherwise)
import socket
bootstrap = os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092")
host, port = bootstrap.split(":")
try:
    with socket.create_connection((host, int(port)), timeout=3):
        print("✅ Kafka broker reachable at", bootstrap)
    try:
        from kafka import KafkaAdminClient
        admin = KafkaAdminClient(bootstrap_servers=bootstrap, request_timeout_ms=5000)
        print("   topics:", sorted(admin.list_topics())); admin.close()
    except Exception as e:
        print("   (could not list topics:", type(e).__name__, e, ")")
except OSError as e:
    print(f"⚠️ Kafka not reachable at {bootstrap} ({type(e).__name__}). "
          "Fine if you are not using Docker; needed only for the Kafka part of Lab 3.")

In [ ]:
# optional libraries used in specific labs
import importlib
for mod, lab in [("pandas", "all"), ("pyarrow", "Lab 2"), ("matplotlib", "Labs 1-5"),
                 ("networkx", "Lab 4"), ("kafka", "Lab 3"), ("qiskit", "Lab 5"), ("qiskit_aer", "Lab 5")]:
    try:
        m = importlib.import_module(mod); print(f"✅ {mod:<11} {getattr(m, '__version__', '?'):<10} ({lab})")
    except ImportError:
        print(f"❌ {mod:<11} missing    ({lab}) -> pip install {mod.replace('_', '-')}")

In [ ]:
spark.stop()
print("All done — your environment is ready for HY562 labs.")